# 🎬 Content-Based Movie Recommendation System
### TMDB 5000 Movies Dataset

---

**Features Implemented:**
- ✅ Data loading, cleaning & preprocessing
- ✅ TF-IDF vectorization
- ✅ Cosine similarity matrix
- ✅ `recommend()` function with Top-N results & similarity scores
- ✅ Recommendation explanation (genre, director, keyword matching)
- ✅ Fuzzy search for partial/misspelled movie names
- ✅ Three recommendation modes: Genre | Description | Hybrid
- ✅ Similarity heatmap & network graph visualizations
- ✅ Evaluation section with quality discussion

---

## 📦 1. Imports & Setup

In [ ]:
import sys
!{sys.executable} -m pip install fuzzywuzzy python-Levenshtein thefuzz networkx seaborn --quiet


In [ ]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
try:
    from fuzzywuzzy import process as fuzz_process
except ImportError:
    from thefuzz import process as fuzz_process
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import networkx as nx
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
print("✅ All libraries loaded successfully!")


## 📥 2. Dataset Loading

In [ ]:
movies_df  = pd.read_csv('tmdb_5000_movies.csv', engine='python')
credits_df = pd.read_csv('tmdb_5000_credits.csv', engine='python')
print(f"Movies  dataset : {movies_df.shape[0]:,} rows × {movies_df.shape[1]} columns")
print(f"Credits dataset : {credits_df.shape[0]:,} rows × {credits_df.shape[1]} columns")
print()
print("Movies columns :" , list(movies_df.columns))
print("Credits columns:", list(credits_df.columns))


In [ ]:
movies_df.head(3)


In [ ]:
credits_df.head(3)


In [ ]:
print("── Movies info ──")
movies_df.info()
print()
print("── Missing values (movies) ──")
print(movies_df.isnull().sum()[movies_df.isnull().sum() > 0])


## 🧹 3. Data Cleaning & Preprocessing

In [ ]:
df = movies_df.merge(credits_df, on='title')
print(f"Merged shape: {df.shape}")
KEEP_COLS = ['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
             'vote_average', 'vote_count', 'popularity', 'release_date']
df = df[KEEP_COLS].copy()
df.head(2)


In [ ]:
def parse_list_col(x, key='name', limit=None):
    """Extract `key` values from a JSON-like list string."""
    try:
        items = [d[key] for d in ast.literal_eval(x)]
        return items[:limit] if limit else items
    except (ValueError, TypeError, KeyError):
        return []
def parse_director(crew_str):
    """Return [director_name] from the crew JSON string."""
    try:
        for d in ast.literal_eval(crew_str):
            if d.get('job') == 'Director':
                return [d['name']]
    except (ValueError, TypeError):
        pass
    return []
def clean_token(name):
    """Remove spaces & lowercase so multi-word names become single tokens."""
    return name.lower().replace(' ', '')
df['genres_list']   = df['genres'].apply(parse_list_col)
df['keywords_list'] = df['keywords'].apply(parse_list_col)
df['cast_list']     = df['cast'].apply(lambda x: parse_list_col(x, limit=4))
df['director_list'] = df['crew'].apply(parse_director)
df['overview'] = df['overview'].fillna('')
print(f"Rows with no overview: {(df['overview'] == '').sum()}")
print(f"Rows with no director: {df['director_list'].apply(len).eq(0).sum()}")
print(f"Sample genres : {df['genres_list'].iloc[0]}")
print(f"Sample cast   : {df['cast_list'].iloc[0]}")
print(f"Sample director: {df['director_list'].iloc[0]}")


In [ ]:
df['genres_tokens']   = df['genres_list'].apply(lambda lst: [clean_token(g) for g in lst])
df['keywords_tokens'] = df['keywords_list'].apply(lambda lst: [clean_token(k) for k in lst])
df['cast_tokens']     = df['cast_list'].apply(lambda lst: [clean_token(c) for c in lst])
df['director_tokens'] = df['director_list'].apply(lambda lst: [clean_token(d) for d in lst])
df['overview_tokens'] = df['overview'].apply(
    lambda s: s.lower().replace(',', ' ').replace('.', ' ').split()
)
df['text_genre'] = df['genres_tokens'].apply(lambda t: ' '.join(t * 3))
df['text_desc'] = df['overview'].str.lower()
df['text_hybrid'] = df.apply(lambda row: ' '.join(
    row['genres_tokens'] * 3 +
    row['keywords_tokens'] +
    row['cast_tokens'] +
    row['director_tokens'] * 3 +
    row['overview_tokens']
), axis=1)
df.reset_index(drop=True, inplace=True)
print("✅ Feature engineering complete!")
print(f"Total usable movies: {len(df):,}")
print()
print("Sample hybrid text for 'Avatar':")
print(df[df['title'] == 'Avatar']['text_hybrid'].values[0][:300], '...')


In [ ]:
print("Cleaned dataset sample:")
df[['title', 'genres_list', 'director_list', 'cast_list', 'keywords_list']].head(5)


## 📊 4. Vectorization

In [ ]:
TFIDF_PARAMS = dict(
    stop_words='english',
    max_features=10_000,
    ngram_range=(1, 2),
    min_df=2
)
tfidf_genre  = TfidfVectorizer(**{**TFIDF_PARAMS, 'max_features': 2000, 'min_df': 1})
tfidf_desc   = TfidfVectorizer(**TFIDF_PARAMS)
tfidf_hybrid = TfidfVectorizer(**{**TFIDF_PARAMS, 'min_df': 1})
mat_genre  = tfidf_genre.fit_transform(df['text_genre'])
mat_desc   = tfidf_desc.fit_transform(df['text_desc'])
mat_hybrid = tfidf_hybrid.fit_transform(df['text_hybrid'])
print(f"Genre  matrix  : {mat_genre.shape}  |  features: {mat_genre.shape[1]:,}")
print(f"Desc   matrix  : {mat_desc.shape}  |  features: {mat_desc.shape[1]:,}")
print(f"Hybrid matrix  : {mat_hybrid.shape}  |  features: {mat_hybrid.shape[1]:,}")


## 🔍 5. Cosine Similarity Matrix

In [ ]:
print("Computing similarity matrices (this may take ~10 s)...")
sim_genre  = cosine_similarity(mat_genre)
sim_desc   = cosine_similarity(mat_desc)
sim_hybrid = cosine_similarity(mat_hybrid)
print(f"Genre  similarity matrix shape : {sim_genre.shape}")
print(f"Desc   similarity matrix shape : {sim_desc.shape}")
print(f"Hybrid similarity matrix shape : {sim_hybrid.shape}")
title_to_idx = pd.Series(df.index, index=df['title']).to_dict()
all_titles   = df['title'].tolist()
print(f"\n✅ Similarity matrices ready. Total movies indexed: {len(title_to_idx):,}")


## 🎯 6. Recommendation Engine

In [ ]:
def fuzzy_find(query, n=5, score_cutoff=50):
    """
    Fuzzy-search movie titles.
    Returns list of (matched_title, score) tuples.
    """
    results = fuzz_process.extractBests(query, all_titles,
                                        limit=n, score_cutoff=score_cutoff)
    return results
def explain_recommendation(source_idx, target_idx):
    """
    Build a human-readable explanation for why `target` was recommended
    given `source`.
    """
    src = df.iloc[source_idx]
    tgt = df.iloc[target_idx]
    reasons = []
    shared_genres = set(src['genres_list']) & set(tgt['genres_list'])
    if shared_genres:
        reasons.append(f"Genres: {', '.join(sorted(shared_genres))}")
    shared_dir = set(src['director_list']) & set(tgt['director_list'])
    if shared_dir:
        reasons.append(f"Director: {', '.join(shared_dir)}")
    shared_cast = set(src['cast_list']) & set(tgt['cast_list'])
    if shared_cast:
        reasons.append(f"Cast: {', '.join(list(shared_cast)[:3])}")
    shared_kw = set(src['keywords_list']) & set(tgt['keywords_list'])
    if shared_kw:
        reasons.append(f"Keywords: {', '.join(list(shared_kw)[:3])}")
    return ' | '.join(reasons) if reasons else 'Similar thematic content'
def recommend(movie_name, top_n=5, mode='hybrid', explain=True, verbose=True):
    """
    Recommend top-N movies similar to `movie_name`.
    Parameters
    ----------
    movie_name : str  — exact or partial title
    top_n      : int  — number of recommendations (default 5)
    mode       : str  — 'hybrid' | 'genre' | 'description'
    explain    : bool — show explanation column
    verbose    : bool — print formatted table
    Returns
    -------
    pd.DataFrame with columns: Rank, Movie, Score, Explanation
    """
    if movie_name not in title_to_idx:
        matches = fuzzy_find(movie_name, n=5)
        if not matches:
            print(f"❌ '{movie_name}' not found and no fuzzy matches either.")
            return None
        print(f"🔍 '{movie_name}' not found exactly. Did you mean:")
        for m_title, m_score in matches:
            print(f"   → {m_title}  (match score: {m_score})")
        movie_name = matches[0][0]
        print(f"   ✅ Using: '{movie_name}'\n")
    idx = title_to_idx[movie_name]
    mode = mode.lower()
    if mode == 'genre':
        sim_matrix = sim_genre
        mode_label = 'Genre-Based'
    elif mode == 'description':
        sim_matrix = sim_desc
        mode_label = 'Description-Based'
    else:
        sim_matrix = sim_hybrid
        mode_label = 'Hybrid'
    scores = list(enumerate(sim_matrix[idx]))
    scores = [(i, s) for i, s in scores if i != idx]
    scores.sort(key=lambda x: x[1], reverse=True)
    top = scores[:top_n]
    rows = []
    for rank, (i, score) in enumerate(top, 1):
        row = {
            'Rank'       : rank,
            'Movie'      : df.iloc[i]['title'],
            'Score'      : round(float(score), 4),
            'Rating'     : df.iloc[i]['vote_average'],
        }
        if explain:
            row['Why Recommended'] = explain_recommendation(idx, i)
        rows.append(row)
    result = pd.DataFrame(rows)
    if verbose:
        src_row = df.iloc[idx]
        print("═" * 72)
        print(f"  🎬  Recommendations for: {movie_name}")
        print(f"  📂  Mode: {mode_label}")
        print(f"  🎭  Genres : {', '.join(src_row['genres_list']) or 'N/A'}")
        print(f"  🎥  Director: {', '.join(src_row['director_list']) or 'N/A'}")
        print(f"  ⭐  Rating : {src_row['vote_average']}")
        print("═" * 72)
        for _, row in result.iterrows():
            bar = '█' * int(row['Score'] * 30)
            print(f"  {int(row['Rank']):>2}. {row['Movie']:<40} {row['Score']:.4f}  {bar}")
            if explain and 'Why Recommended' in row:
                print(f"      💡 {row['Why Recommended']}")
        print("═" * 72)
    return result
print("✅ Recommendation engine ready! Functions: recommend(), fuzzy_find()")


## 🧪 7. Test Recommendations

In [ ]:
r = recommend('The Dark Knight', top_n=7, mode='hybrid')


In [ ]:
r = recommend('Inception', top_n=5, mode='genre')


In [ ]:
r = recommend('Interstellar', top_n=5, mode='description')


## 🔍 8. Fuzzy Search System

In [ ]:
print("🔍 Fuzzy Search Demo")
print("═" * 50)
queries = ['bat', 'interstela', 'spidermn', 'star war', 'avngrs']
for q in queries:
    matches = fuzzy_find(q, n=3)
    print(f"  Query: '{q}'")
    for title, score in matches:
        print(f"    → {title}  (score: {score})")
    print()


In [ ]:
r = recommend('nolan dark knight', top_n=5)


## 🎭 9. Three Recommendation Modes — Side-by-Side

In [ ]:
def compare_modes(movie_name, top_n=5):
    """Show recommendations from all three modes side by side."""
    genre_recs = recommend(movie_name, top_n=top_n, mode='genre',       explain=False, verbose=False)
    desc_recs  = recommend(movie_name, top_n=top_n, mode='description', explain=False, verbose=False)
    hyb_recs   = recommend(movie_name, top_n=top_n, mode='hybrid',      explain=False, verbose=False)
    if genre_recs is None:
        return
    comparison = pd.DataFrame({
        '🎭 Genre-Based'      : genre_recs['Movie'].values,
        '📝 Description-Based': desc_recs['Movie'].values,
        '🔀 Hybrid (Best)'    : hyb_recs['Movie'].values,
    }, index=[f"#{i}" for i in range(1, top_n + 1)])
    print(f"\n🎬 Mode comparison for: {movie_name}")
    print(comparison.to_string())
    return comparison
compare_modes('Inception', top_n=5)


In [ ]:
compare_modes('The Avengers', top_n=5)


## 📈 10. Visualizations

### 10a. Similarity Heatmap (Top-10 Similar Movies)

In [ ]:
def plot_heatmap(movie_name, top_n=10, mode='hybrid', figsize=(13, 11)):
    """Plot a cosine similarity heatmap for a movie and its top-N neighbours."""
    if movie_name not in title_to_idx:
        matches = fuzzy_find(movie_name, n=1)
        if not matches: return
        movie_name = matches[0][0]
    sim_matrix = sim_hybrid if mode == 'hybrid' else (sim_genre if mode == 'genre' else sim_desc)
    idx = title_to_idx[movie_name]
    scores = sorted(enumerate(sim_matrix[idx]), key=lambda x: x[1], reverse=True)
    top_indices = [idx] + [i for i, _ in scores if i != idx][:top_n]
    labels      = [df.iloc[i]['title'][:30] for i in top_indices]
    sub_matrix = sim_matrix[np.ix_(top_indices, top_indices)]
    fig, ax = plt.subplots(figsize=figsize)
    mask = np.eye(len(top_indices), dtype=bool)
    sns.heatmap(
        sub_matrix,
        annot=True, fmt='.2f', cmap='YlOrRd',
        xticklabels=labels, yticklabels=labels,
        linewidths=0.5, linecolor='#cccccc',
        vmin=0, vmax=1, ax=ax,
        annot_kws={'size': 8}
    )
    ax.set_title(f'Cosine Similarity Heatmap\n"{movie_name}" — Top {top_n} Similar Movies ({mode.title()} Mode)',
                 fontsize=13, fontweight='bold', pad=15)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
    ax.add_patch(plt.Rectangle((0, 0), len(top_indices), 1, fill=False,
                                edgecolor='royalblue', lw=3))
    ax.add_patch(plt.Rectangle((0, 0), 1, len(top_indices), fill=False,
                                edgecolor='royalblue', lw=3))
    plt.tight_layout()
    plt.savefig(f'heatmap_{movie_name.replace(" ","_")[:20]}.png',
                bbox_inches='tight', dpi=150)
    plt.show()
    print(f"✅ Heatmap saved.")
plot_heatmap('Inception', top_n=10)


### 10b. Similarity Network Graph

In [ ]:
def plot_network(movie_name, top_n=10, mode='hybrid', threshold=0.05, figsize=(14, 10)):
    """
    Draw a network graph where:
    - nodes  = the query movie + top-N similar movies
    - edges  = cosine similarity (only above `threshold`)
    - edge weight / thickness reflects similarity strength
    """
    if movie_name not in title_to_idx:
        matches = fuzzy_find(movie_name, n=1)
        if not matches: return
        movie_name = matches[0][0]
    sim_matrix = sim_hybrid if mode == 'hybrid' else (sim_genre if mode == 'genre' else sim_desc)
    idx = title_to_idx[movie_name]
    scores = sorted(enumerate(sim_matrix[idx]), key=lambda x: x[1], reverse=True)
    top_indices = [idx] + [i for i, _ in scores if i != idx][:top_n]
    labels = {i: df.iloc[i]['title'][:22] for i in top_indices}
    node_scores = {i: (sim_matrix[idx][i] if i != idx else 1.0) for i in top_indices}
    G = nx.Graph()
    for i in top_indices:
        G.add_node(i, label=labels[i], score=node_scores[i])
    for i in top_indices:
        for j in top_indices:
            if i < j:
                w = sim_matrix[i][j]
                if w > threshold:
                    G.add_edge(i, j, weight=float(w))
    pos = nx.spring_layout(G, seed=42, k=2.0)
    node_colors = ['#FFD700' if i == idx else
                   plt.cm.Blues(0.3 + 0.7 * node_scores[i]) for i in G.nodes()]
    node_sizes  = [3000 if i == idx else 800 + 2000 * node_scores[i] for i in G.nodes()]
    edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
    edge_widths  = [1 + 6 * w for w in edge_weights]
    edge_alphas  = [0.3 + 0.7 * w for w in edge_weights]
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor('#0d1117')
    fig.patch.set_facecolor('#0d1117')
    nx.draw_networkx_edges(G, pos, ax=ax,
                           width=edge_widths,
                           alpha=0.6,
                           edge_color='#4a9eff')
    nx.draw_networkx_nodes(G, pos, ax=ax,
                           node_color=node_colors,
                           node_size=node_sizes,
                           alpha=0.95)
    nx.draw_networkx_labels(G, pos, labels={i: labels[i] for i in G.nodes()},
                            ax=ax, font_size=7.5, font_color='white', font_weight='bold')
    edge_labels = {(u, v): f"{G[u][v]['weight']:.2f}" for u, v in G.edges()
                   if G[u][v]['weight'] > 0.15}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                                 font_size=6, font_color='#aaddff', ax=ax)
    ax.set_title(f'Similarity Network Graph — "{movie_name}" ({mode.title()} Mode)\n'
                 f'Gold node = query movie  |  Blue nodes = recommendations  |  '
                 f'Edge thickness = similarity',
                 color='white', fontsize=11, fontweight='bold', pad=12)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'network_{movie_name.replace(" ","_")[:20]}.png',
                bbox_inches='tight', dpi=150, facecolor=fig.get_facecolor())
    plt.show()
    print(f"✅ Network graph saved. Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
plot_network('The Dark Knight', top_n=10)


In [ ]:
from collections import Counter
all_genres = [g for gl in df['genres_list'] for g in gl]
genre_counts = Counter(all_genres).most_common(15)
g_names, g_vals = zip(*genre_counts)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(g_names[::-1], g_vals[::-1],
               color=plt.cm.viridis(np.linspace(0.2, 0.85, len(g_names))))
for bar, val in zip(bars, g_vals[::-1]):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
ax.set_title('Genre Distribution in TMDB 5000 Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Movies')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('genre_distribution.png', bbox_inches='tight', dpi=150)
plt.show()


## 📋 11. Evaluation Section

In [ ]:
eval_movies = ['Inception', 'Interstellar', 'The Dark Knight']
eval_results = {}
for movie in eval_movies:
    print(f"\n{'─'*68}")
    res = recommend(movie, top_n=5, mode='hybrid', explain=True)
    eval_results[movie] = res


In [ ]:
discussion = """
╔══════════════════════════════════════════════════════════════════════════╗
║              📊  EVALUATION: QUALITY DISCUSSION                         ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  🎬 Inception                                                            ║
║    ✅ Works well: Other Christopher Nolan films rank very high          ║
║       (The Dark Knight, Interstellar, The Prestige) sharing director    ║
║       and cast (DiCaprio, Cillian Murphy, Ken Watanabe).                ║
║    ✅ Thematic overlap (dreams, heist, mind-bending) correctly picked.  ║
║    ⚠️  Limitation: shallow semantic overlap misses some great sci-fi    ║
║       thrillers that aren't keyword-rich in the dataset.                ║
║                                                                          ║
║  🌌 Interstellar                                                         ║
║    ✅ Works well: Space-travel, sci-fi, Nolan-directed films surface.   ║
║    ✅ 2001: A Space Odyssey / Gravity / The Martian recommended well.   ║
║    ⚠️  Limitation: Drama/emotional-depth scores are implicit in TF-IDF; ║
║       purely keyword-driven matching can be imprecise for quiet films.  ║
║                                                                          ║
║  🦇 The Dark Knight                                                      ║
║    ✅ Works very well: Superhero/action + Nolan makes it easy to score  ║
║       Batman Begins and other DC/superhero films correctly.             ║
║    ✅ The Joker keyword links to darker, crime-thriller adjacent films.  ║
║    ⚠️  Limitation: Marvel vs DC nuance is lost; both treated as         ║
║       'superhero' by the vectorizer.                                    ║
║                                                                          ║
║  🔑 What works well overall:                                             ║
║    • Director + cast as weighted tokens is very effective               ║
║    • Genre + keyword repetition weighting improves precision            ║
║    • Fuzzy search makes the system very user-friendly                   ║
║                                                                          ║
║  🚀 What could be improved:                                              ║
║    • Add a neural text embedding (BERT/Sentence-Transformers)           ║
║      for richer semantic understanding of overviews                     ║
║    • Incorporate user ratings (hybrid CF + CBF)                         ║
║    • Add temporal decay so older films aren't over-represented          ║
║    • Synonym expansion (e.g. "space" ↔ "cosmos" ↔ "galaxy")           ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
print(discussion)


## 🎛️ 12. Interactive Recommender (Quick-Use Interface)

In [ ]:
MOVIE_NAME = "The Matrix"
TOP_N      = 8
MODE       = 'hybrid'
results = recommend(MOVIE_NAME, top_n=TOP_N, mode=MODE, explain=True)
plot_heatmap(MOVIE_NAME, top_n=10, mode=MODE)
plot_network(MOVIE_NAME, top_n=10, mode=MODE)


---

## 🏁 Summary

| Component | Details |
|---|---|
| **Dataset** | TMDB 5000 Movies + Credits (~4,800 movies) |
| **Vectorizer** | TF-IDF with bi-grams, 10K features |
| **Similarity** | Cosine similarity |
| **Features used** | Genres × 3 + Keywords + Cast (top 4) + Director × 3 + Overview |
| **Modes** | Hybrid (default), Genre-only, Description-only |
| **Fuzzy Search** | fuzzywuzzy token-set ratio |
| **Explanations** | Shared genres, director, cast, keywords |
| **Visualizations** | Heatmap + Network graph + Genre distribution |

---
*Built with scikit-learn, fuzzywuzzy, networkx, seaborn, matplotlib.*